# Upstream Network Analysis

This notebook demonstrates HMS basin network topology analysis using reproducible example-project folders.

It first checks the local Spring Creek eBFE delivery under `H:\Testing\eBFE Model Organization` because that project is useful local engineering context. The delivered Spring Creek eBFE model is RAS-only and does not contain an HMS basin model, so the executable HMS network demonstration uses the packaged `tenk` HMS sample instead.

## What You'll Learn

1. Confirming whether a candidate project contains HMS basin inputs
2. Extracting HMS element types from a reproducible sample project
3. Building reverse network lookup tables
4. Finding all upstream elements for a target
5. Calculating contributing drainage area
6. Exporting GIS layers for spatial verification

## Prerequisites

- hms-commander installed with GIS extras
- HEC-HMS sample projects available through `HmsExamples`
- Optional local Spring eBFE project folder for the availability check


In [ ]:
# pip install "hms-commander[dss,gis]" geopandas matplotlib


## Setup


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
repo_root = cwd if (cwd / "hms_commander").exists() else cwd.parent
if (repo_root / "hms_commander").exists() and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from hms_commander import HmsBasin, HmsGeo, HmsExamples, __version__

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

NOTEBOOK_ID = "015"
example_root = Path.cwd() / "example_projects"
workspace = example_root / f"upstream_network_{NOTEBOOK_ID}"
workspace.mkdir(parents=True, exist_ok=True)

print(f"hms-commander v{__version__}")
print(f"Repository root: {repo_root}")
print(f"Example root: {example_root}")


## Helper Functions


In [ ]:
def summarize_basin(path: Path) -> dict:
    subbasins = HmsBasin.get_subbasins(path)
    junctions = HmsBasin.get_junctions(path)
    reaches = HmsBasin.get_reaches(path)
    diversions = HmsBasin.get_diversions(path)
    reservoirs = HmsBasin.get_reservoirs(path)
    sinks = HmsBasin.get_sinks(path)
    network = HmsBasin.get_upstream_network(path)
    return {
        "path": path,
        "Basin": path.name,
        "Subbasins": len(subbasins),
        "Junctions": len(junctions),
        "Reaches": len(reaches),
        "Diversions": len(diversions),
        "Reservoirs": len(reservoirs),
        "Sinks": len(sinks),
        "Network_Targets": len(network),
        "Network_Links": sum(len(values) for values in network.values()),
        "Total_Area_sqmi": float(subbasins["area"].sum()) if "area" in subbasins else 0.0,
    }


def summarize_project(project_dir: Path) -> pd.DataFrame:
    basin_files = sorted(project_dir.glob("*.basin"))
    if not basin_files:
        raise FileNotFoundError(f"No .basin files found in {project_dir}")
    return pd.DataFrame(summarize_basin(path) for path in basin_files)


## Spring eBFE Project Check

The Spring Creek eBFE delivery is the preferred local engineering reference for this notebook review, but it is a RAS-only Pattern 3a delivery. The check below verifies whether it contains any HMS project files before the notebook moves on to the packaged HMS sample used for topology analysis.


In [ ]:
spring_ebfe_root = Path(r"H:\Testing\eBFE Model Organization\Organized\SpringCreek_12040102")
spring_hms_dir = spring_ebfe_root / "HMS Model"

spring_hms_files = sorted(spring_hms_dir.rglob("*.hms")) if spring_hms_dir.exists() else []
spring_basin_files = sorted(spring_hms_dir.rglob("*.basin")) if spring_hms_dir.exists() else []

spring_status = pd.DataFrame([
    {
        "Project": "SpringCreek_12040102",
        "Root Exists": spring_ebfe_root.exists(),
        "HMS Model Folder Exists": spring_hms_dir.exists(),
        "HMS Project Files": len(spring_hms_files),
        "Basin Files": len(spring_basin_files),
        "Path": str(spring_ebfe_root),
    }
])

display(spring_status)

if not spring_ebfe_root.exists():
    print("Spring eBFE folder is not available in this environment; continuing with packaged HMS sample.")
elif not spring_hms_files and not spring_basin_files:
    print("Spring eBFE is present but RAS-only for this delivery; continuing with packaged HMS sample.")
else:
    print("Spring eBFE contains HMS files and can be used for HMS-specific follow-up analysis.")


## Network Demonstration Project

Because the Spring Creek eBFE delivery does not include HMS basin inputs, the executable upstream-network workflow uses HMS sample project `tenk`. It has a small but complete routing network and is extracted into a notebook-scoped workspace under `example_projects/upstream_network_015/`.


In [ ]:
network_project_dir = HmsExamples.extract_project(
    "tenk",
    output_path=workspace / "standard_hms",
    overwrite=False,
)

network_summary = summarize_project(network_project_dir).sort_values(
    ["Network_Links", "Subbasins"],
    ascending=False,
).reset_index(drop=True)

basin_path = Path(network_summary.loc[0, "path"])
geo_files = sorted(network_project_dir.glob("*.geo"))
map_files = sorted(network_project_dir.glob("*.map"))
geo_path = geo_files[0] if geo_files else None
map_path = map_files[0] if map_files else None

print(f"Network project folder: {network_project_dir}")
print(f"Selected basin: {basin_path.name}")
print(f"Geometry file: {geo_path.name if geo_path else 'not found'}")
print(f"Map file: {map_path.name if map_path else 'not found'}")
display(network_summary.drop(columns=["path"]))


## 1. Extract HMS Element Types

A complete topology pass starts by reading subbasins, junctions, reaches, diversions, reservoirs, and sinks. This public sample does not contain diversions, but the same calls handle diversion-bearing projects.


In [ ]:
subbasins = HmsBasin.get_subbasins(basin_path)
junctions = HmsBasin.get_junctions(basin_path)
reaches = HmsBasin.get_reaches(basin_path)
diversions = HmsBasin.get_diversions(basin_path)
reservoirs = HmsBasin.get_reservoirs(basin_path)
sinks = HmsBasin.get_sinks(basin_path)

print(f"Subbasins:  {len(subbasins)}")
print(f"Junctions:  {len(junctions)}")
print(f"Reaches:    {len(reaches)}")
print(f"Diversions: {len(diversions)}")
print(f"Reservoirs: {len(reservoirs)}")
print(f"Sinks:      {len(sinks)}")

if diversions.empty:
    print("No diversions found in this basin.")
else:
    display(diversions[["name", "downstream", "divert_to", "canvas_x", "canvas_y"]])


## 2. Build Reverse Network Lookup

HMS basin files store downstream connections. `HmsBasin.get_upstream_network()` inverts those relationships into a lookup of what flows into each target element.


In [ ]:
upstream_network = HmsBasin.get_upstream_network(basin_path)

print(f"Targets with upstream connections: {len(upstream_network)}")
print(f"Total upstream links: {sum(len(values) for values in upstream_network.values())}")

for candidate, direct in list(upstream_network.items())[:10]:
    print(f"{candidate}: {len(direct)} direct upstream element(s)")


## 3. Select a Target

Pick the element with the largest contributing area and at least one upstream subbasin.


In [ ]:
target_rows = []
for candidate in sorted(upstream_network):
    upstream_candidate = HmsBasin.get_upstream_elements(basin_path, candidate)
    if not upstream_candidate["subbasins"]:
        continue

    area_sqmi = HmsBasin.get_contributing_area(basin_path, candidate)
    target_rows.append({
        "Target": candidate,
        "Area_sqmi": area_sqmi,
        "Upstream_Subbasins": len(upstream_candidate["subbasins"]),
        "Upstream_Junctions": len(upstream_candidate["junctions"]),
        "Upstream_Reaches": len(upstream_candidate["reaches"]),
        "Upstream_Diversions": len(upstream_candidate["diversions"]),
    })

if not target_rows:
    raise RuntimeError("No target with upstream subbasins was found.")

targets_df = pd.DataFrame(target_rows).sort_values(
    ["Area_sqmi", "Upstream_Subbasins"],
    ascending=False,
).reset_index(drop=True)

target = targets_df.loc[0, "Target"]
print(f"Selected target: {target}")
display(targets_df.head(10))


## 4. Find All Upstream Elements

`HmsBasin.get_upstream_elements()` recursively traverses the reverse network and returns upstream elements grouped by HMS element type.


In [ ]:
upstream = HmsBasin.get_upstream_elements(basin_path, target)

print(f"=== Upstream Elements for {target} ===")
print(f"Subbasins:  {len(upstream['subbasins'])}")
print(f"Junctions:  {len(upstream['junctions'])}")
print(f"Reaches:    {len(upstream['reaches'])}")
print(f"Diversions: {len(upstream['diversions'])}")
print(f"Total:      {sum(len(values) for values in upstream.values())}")

print("\nUpstream subbasins:")
for name in upstream["subbasins"]:
    print(f"  - {name}")


## 5. Calculate Contributing Drainage Area

`HmsBasin.get_contributing_area()` combines upstream traversal with subbasin area summation.


In [ ]:
area = HmsBasin.get_contributing_area(basin_path, target)
total_area = float(subbasins["area"].sum()) if "area" in subbasins else 0.0

print(f"=== Contributing Drainage Area for {target} ===")
print(f"Target contributing area: {area:.2f} sq mi")
print(f"Total basin subbasin area: {total_area:.2f} sq mi")
print(f"Upstream subbasins: {len(upstream['subbasins'])} of {len(subbasins)}")
print(f"Upstream diversions: {len(upstream['diversions'])} of {len(diversions)}")


## 6. Extract GIS Layers

The Tenk sample uses HMS schematic coordinates rather than a real-world projection. `EPSG:3857` is used here only as a convenient plotting CRS for the exported GeoJSON layers.


In [ ]:
output_dir = workspace / "outputs" / basin_path.stem
output_dir.mkdir(parents=True, exist_ok=True)

outputs = HmsGeo.extract_all_gis(
    basin_path=basin_path,
    geo_path=geo_path if geo_path and geo_path.exists() else None,
    map_path=map_path if map_path and map_path.exists() else None,
    output_dir=output_dir,
    crs_epsg="EPSG:3857",
    include_diversions=True,
)

print("=== Extracted GeoJSON Files ===")
for key, path in outputs.items():
    if path.exists():
        gdf = gpd.read_file(path)
        print(f"{key:15} -> {path.name:30} ({len(gdf)} features)")


## 7. Visualize the Upstream Network


In [ ]:
subbasins_gdf = gpd.read_file(outputs["subbasins"])
junctions_gdf = gpd.read_file(outputs["junctions"])
reaches_gdf = gpd.read_file(outputs["reaches"]) if "reaches" in outputs else None
diversions_gdf = gpd.read_file(outputs["diversions"]) if "diversions" in outputs else None

upstream_names = set(upstream["subbasins"])
subbasins_gdf["is_upstream"] = subbasins_gdf["name"].isin(upstream_names)

fig, ax = plt.subplots(figsize=(10, 8))

other_subbasins = subbasins_gdf[~subbasins_gdf["is_upstream"]]
upstream_subbasins = subbasins_gdf[subbasins_gdf["is_upstream"]]

if not other_subbasins.empty:
    other_subbasins.plot(
        ax=ax,
        color="lightgray",
        alpha=0.5,
        label="Other subbasins",
    )
if not upstream_subbasins.empty:
    upstream_subbasins.plot(
        ax=ax,
        color="skyblue",
        alpha=0.75,
        edgecolor="blue",
        linewidth=1.0,
        label=f"Upstream subbasins ({len(upstream_names)})",
    )

if reaches_gdf is not None and not reaches_gdf.empty:
    reaches_gdf.plot(ax=ax, color="black", linewidth=1.5, alpha=0.6, zorder=2)

if not junctions_gdf.empty:
    junctions_gdf.plot(ax=ax, color="gray", markersize=35, alpha=0.7, zorder=3)

if diversions_gdf is not None and not diversions_gdf.empty:
    diversions_gdf.plot(
        ax=ax,
        color="red",
        marker="D",
        markersize=80,
        alpha=0.8,
        zorder=4,
        label=f"Diversions ({len(diversions_gdf)})",
    )

target_junction = junctions_gdf[junctions_gdf["name"] == target]
if not target_junction.empty:
    target_junction.plot(
        ax=ax,
        color="green",
        marker="*",
        markersize=350,
        zorder=5,
        label=f"Target: {target}",
    )

ax.set_title(
    f"Upstream Network Analysis: {target}\n"
    f"Contributing Area: {area:.2f} sq mi from {len(upstream_names)} subbasins",
    fontsize=13,
    fontweight="bold",
)
ax.set_xlabel("Schematic X")
ax.set_ylabel("Schematic Y")
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
figure_path = output_dir / f"upstream_network_{target}.png"
plt.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Map saved to: {figure_path}")


## 8. Compare Candidate Targets


In [ ]:
print("=== Top Candidate Targets by Contributing Area ===")
display(targets_df)

print("=== Summary Statistics ===")
print(f"Targets analyzed: {len(targets_df)}")
print(f"Area range: {targets_df['Area_sqmi'].min():.2f} - {targets_df['Area_sqmi'].max():.2f} sq mi")
print(f"Mean area: {targets_df['Area_sqmi'].mean():.2f} sq mi")
print(f"Median area: {targets_df['Area_sqmi'].median():.2f} sq mi")
print(f"Targets with upstream diversions: {(targets_df['Upstream_Diversions'] > 0).sum()}")


## Key Takeaways

1. Candidate local eBFE folders should be checked for HMS inputs before using them in HMS-specific examples.
2. Spring Creek eBFE is present in the local engineering workspace, but this delivery is RAS-only and has no HMS basin file.
3. `HmsExamples.extract_project(..., overwrite=False)` provides a reproducible HMS fixture without hard-coded local project paths.
4. `get_upstream_network()`, `get_upstream_elements()`, and `get_contributing_area()` work on the selected network basin without local-path dependencies.
5. `extract_all_gis(include_diversions=True)` supports spatial verification and includes diversion layers when the basin contains diversions.
